In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,  classification_report
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from scipy.optimize import minimize
from sklearn.metrics import f1_score


In [2]:
# Carregando os dados
df =  pd.read_csv("../Tratamento de Dados/dataset_tratado.csv")

In [3]:
def Modelagem(X_train, y_train, X_test, Modelo):
    
    pca_pipeline = Pipeline([
        ('scaler', StandardScaler()),        
        ('pca', PCA(n_components=0.95)),      
        ('modelo', Modelo())   
    ])
    
    pca_pipeline.fit(X_train, y_train)
    
    previsoes = pca_pipeline.predict(X_test)

    return previsoes

In [4]:
df.drop('Unnamed: 0', inplace = True, axis = 1)

In [5]:
df.head()

,Age,CDR,SES,nWBV,Visit,MMSE,ASF,eTIV,Group,MR Delay
0,87.0,0.0,2.0,0.696106,1.0,27.0,0.883440,1986.550000,0.0,0.0
1,88.0,0.0,2.0,0.681062,2.0,30.0,0.875539,2004.479526,0.0,457.0
2,75.0,0.5,1.8,0.736336,1.0,23.0,1.045710,1678.290000,1.0,0.0
3,76.0,0.5,1.6,0.713402,2.0,28.0,1.010000,1737.620000,1.0,560.0
4,80.0,0.5,2.6,0.701236,3.0,22.0,1.033623,1697.911134,1.0,1895.0


In [6]:
X = df.drop('Group', axis = 1)
y = df['Group']

## **Separando os dados em treino e teste**

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## **Treinando o modelo**

In [ ]:
predicao = Modelagem(X_train, y_train, X_test, SVC)
print(classification_report(y_test, predicao))

              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75



Essa foi a tabela de métricas alcançada pelos pesquisadores do artigo.

# **Mudando os métodos de otimização**

### Função de Loss — Squared Hinge SVM

$$L(w) = \frac{1}{2}||w||^2 + C \sum_{i=1}^{N} \max(0, 1 - y_i \cdot x_i^T w)^2$$

**Onde:**
- $w$ — vetor de pesos (o que o modelo aprende)
- $C$ — regularização: controla o trade-off entre margem e erros
- $y_i \in \{-1, +1\}$ — rótulo da amostra $i$
- $x_i$ — vetor de features da amostra $i$
- $m_i = y_i \cdot x_i^T w$ — margem da amostra $i$

---

### Gradiente

$$\nabla L(w) = w - 2C \sum_{i: m_i < 1} (1 - m_i) \cdot y_i x_i$$

**Onde:**
- O somatório considera apenas os pontos que violam a margem ($m_i < 1$)
- Pontos com $m_i \geq 1$ têm gradiente zero — já estão classificados corretamente com folga

---

### Efeito do parâmetro C

| $C$ | Comportamento |
|-----|--------------|
| $C \to 0$ | Margem larga, aceita mais erros — underfitting |
| $C = 1$ | Equilíbrio padrão |
| $C \to \infty$ | Margem estreita, tolera poucos erros — overfitting |

---

### Condição de otimalidade

O ponto $w^*$ é ótimo quando:

$$||\nabla L(w^*)|| \approx 0$$

In [10]:
y.value_counts()

Group
0.0    227
1.0    146
Name: count, dtype: int64

In [11]:
# Transformando os dados para o formato da classificação -1 e 1 para o SVM
y = np.where(y == 0.0, -1, 1)
scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

np.random.seed(42)

In [12]:
class SVMoptimizer:
    def __init__(self, C=1.0, tol=1e-5, maxiter=1000):
        self.C = C
        self.tol = tol
        self.maxiter = maxiter

    def loss(self, w, X, y):
        margem = y * (X @ w)
        hinge = np.maximum(0, 1 - margem) ** 2
        return 0.5 * np.dot(w, w) + self.C * np.sum(hinge)

    def gradient(self, w, X, y):
        margem = y * (X @ w)
        mascara = margem < 1
        diff = 1 - margem[mascara]
        return w - 2 * self.C * (diff * y[mascara]) @ X[mascara]

    def hessiana(self, w, X, y):
        margem = y * (X @ w)
        mascara = margem < 1
        X_mascara = X[mascara]
        return np.eye(len(w)) + 2 * self.C * X_mascara.T @ X_mascara

    def otimizar(self, X, y, w0, methods):
        max_iter_options = {
            'CG':        {'maxiter': self.maxiter},
            'BFGS':      {'maxiter': self.maxiter},
            'Newton-CG': {'maxiter': self.maxiter},
            'L-BFGS-B':  {'maxiter': self.maxiter},
            'TNC':       {'maxfun':  self.maxiter},
            'SLSQP':     {'maxiter': self.maxiter},
        }

        results = {}
        for method in methods:
            iters = []
            start = time.perf_counter()

            res = minimize(
                fun=self.loss,
                x0=w0,
                jac=self.gradient,
                hess=self.hessiana if method == 'Newton-CG' else None,
                args=(X, y),
                method=method,
                callback=lambda w: iters.append(self.loss(w, X, y)),
                tol=self.tol,
                options= max_iter_options[method],

            )

            elapsed = time.perf_counter() - start
            results[method] = {
                'time':      elapsed,
                'loss':      res.fun,
                'iters':     len(iters),
                'converged': res.success,
                'history':   iters,
                'w':         res.x
            }

            print(f"{method:12s} | tempo: {elapsed:.4f}s | loss: {res.fun:.4f} | iterações: {len(iters)} | convergiu: {res.success}")

        return results

    def predict(self, X, w):
        return np.sign(X @ w)
    
    def avaliar(self, X_test, y_test, results):
        for method, data in results.items():
            y_pred = self.predict(X_test, data['w'])
            f1  = f1_score(y_test, y_pred)
            acc = accuracy_score(y_test, y_pred)
            print(f"  {method:12s} | f1: {f1:.4f} | acc: {acc:.4f} | convergiu: {data['converged']}")



In [13]:
methods = ['CG', 'BFGS', 'Newton-CG', 'L-BFGS-B']
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

# Inicialização aleatória dos pesos
w0 = np.random.randn(X_train.shape[1]) * 0.01

for C in C_values:
    print(f"\n--- C = {C} ---")
    svm = SVMoptimizer(C=C, tol=1e-5, maxiter=1000)
    results = svm.otimizar(X_train, y_train, w0, methods)
    svm.avaliar(X_test, y_test, results)



--- C = 0.001 ---
CG           | tempo: 0.0014s | loss: 0.2004 | iterações: 7 | convergiu: True
BFGS         | tempo: 0.0013s | loss: 0.2004 | iterações: 11 | convergiu: True
Newton-CG    | tempo: 0.0005s | loss: 0.2004 | iterações: 5 | convergiu: True
L-BFGS-B     | tempo: 0.0004s | loss: 0.2004 | iterações: 4 | convergiu: True
  CG           | f1: 0.9552 | acc: 0.9600 | convergiu: True
  BFGS         | f1: 0.9552 | acc: 0.9600 | convergiu: True
  Newton-CG    | f1: 0.9552 | acc: 0.9600 | convergiu: True
  L-BFGS-B     | f1: 0.9552 | acc: 0.9600 | convergiu: True

--- C = 0.01 ---
CG           | tempo: 0.0016s | loss: 0.9184 | iterações: 17 | convergiu: True
BFGS         | tempo: 0.0011s | loss: 0.9184 | iterações: 15 | convergiu: True
Newton-CG    | tempo: 0.0008s | loss: 0.9184 | iterações: 8 | convergiu: True
L-BFGS-B     | tempo: 0.0004s | loss: 0.9184 | iterações: 9 | convergiu: True
  CG           | f1: 0.9697 | acc: 0.9733 | convergiu: True
  BFGS         | f1: 0.9697 | acc: 0

In [14]:
# Inicialização com pesos zerados
w0 = np.zeros(X_train.shape[1])

for C in C_values:
    print(f"\n--- C = {C} ---")
    svm = SVMoptimizer(C=C, tol=1e-5, maxiter=1000)
    results = svm.otimizar(X_train, y_train, w0, methods)
    svm.avaliar(X_test, y_test, results)


--- C = 0.001 ---
CG           | tempo: 0.0008s | loss: 0.2004 | iterações: 6 | convergiu: True
BFGS         | tempo: 0.0011s | loss: 0.2004 | iterações: 11 | convergiu: True
Newton-CG    | tempo: 0.0008s | loss: 0.2004 | iterações: 5 | convergiu: True
L-BFGS-B     | tempo: 0.0003s | loss: 0.2004 | iterações: 4 | convergiu: True
  CG           | f1: 0.9552 | acc: 0.9600 | convergiu: True
  BFGS         | f1: 0.9552 | acc: 0.9600 | convergiu: True
  Newton-CG    | f1: 0.9552 | acc: 0.9600 | convergiu: True
  L-BFGS-B     | f1: 0.9552 | acc: 0.9600 | convergiu: True

--- C = 0.01 ---
CG           | tempo: 0.0016s | loss: 0.9184 | iterações: 18 | convergiu: True
BFGS         | tempo: 0.0011s | loss: 0.9184 | iterações: 14 | convergiu: True
Newton-CG    | tempo: 0.0007s | loss: 0.9184 | iterações: 8 | convergiu: True
L-BFGS-B     | tempo: 0.0004s | loss: 0.9184 | iterações: 10 | convergiu: True
  CG           | f1: 0.9697 | acc: 0.9733 | convergiu: True
  BFGS         | f1: 0.9697 | acc: 

Os resultados se mantêm equivalentes para ambas as inicializações porque a squared hinge loss é uma função convexa e isso garante um único mínimo global, independente do ponto de partida. A qualidade das métricas (f1 ≈ 0.95) é explicada pela boa separabilidade linear dos dados